In [1]:
import os

# 1. Override the broken environment variable BEFORE importing PySpark
# Notice we must append "pyspark-shell" at the end for interactive notebooks
os.environ["PYSPARK_SUBMIT_ARGS"] = "--packages org.apache.spark:spark-sql-kafka-0-10_2.13:4.1.0,org.apache.iceberg:iceberg-spark-runtime-4.0_2.13:1.10.0,org.apache.iceberg:iceberg-aws-bundle:1.10.0 pyspark-shell"

from pyspark.sql import SparkSession

# 2. Initialize Spark 
spark = SparkSession.builder \
    .appName("Gold_Layer_Analysis") \
    .config("spark.sql.extensions", "org.apache.iceberg.spark.extensions.IcebergSparkSessionExtensions") \
    .config("spark.sql.catalog.local", "org.apache.iceberg.spark.SparkCatalog") \
    .config("spark.sql.catalog.local.type", "rest") \
    .config("spark.sql.catalog.local.uri", "http://iceberg-rest:8181") \
    .config("spark.sql.catalog.local.io-impl", "org.apache.iceberg.aws.s3.S3FileIO") \
    .config("spark.sql.catalog.local.warehouse", "s3://warehouse/") \
    .config("spark.sql.catalog.local.s3.endpoint", "http://minio:9000") \
    .config("spark.sql.catalog.local.s3.path-style-access", "true") \
    .getOrCreate()

print("Spark Session successfully connected to Iceberg Catalog!")

Spark Session successfully connected to Iceberg Catalog!


In [2]:
print("--- Gold Assignment Efficiency (Sample) ---")
spark.sql("""
    SELECT 
        driver_id,
        driver_rating,
        trips_assigned,
        ROUND(total_distance, 2) as total_distance_mi,
        ROUND(total_revenue, 2) as total_revenue_usd,
        total_idle_seconds,
        ROUND(avg_fare_per_mile, 2) as avg_fare_per_mi,
        is_active
    FROM local.warehouse.gold_assignment_efficiency
    ORDER BY driver_id
    LIMIT 20
""").show(truncate=False)

--- Gold Assignment Efficiency (Sample) ---
+---------+-------------+--------------+-----------------+-----------------+------------------+---------------+---------+
|driver_id|driver_rating|trips_assigned|total_distance_mi|total_revenue_usd|total_idle_seconds|avg_fare_per_mi|is_active|
+---------+-------------+--------------+-----------------+-----------------+------------------+---------------+---------+
|26       |India        |229           |662.64           |3913.3           |6297              |5.91           |true     |
|35       |France       |230           |602.93           |3357.12          |18270             |5.57           |true     |
|43       |Brazil       |230           |684.2            |3870.4           |16499             |5.66           |true     |
|47       |Italy        |230           |668.67           |3653.4           |16096             |5.46           |true     |
|113      |Brazil       |230           |666.68           |3970.9           |15096             |5.96   

In [3]:
print("--- Q: Which driver has the best fare-per-mile ratio? ---")
spark.sql("""
    SELECT 
        driver_id, 
        driver_rating, 
        ROUND(avg_fare_per_mile, 2) AS peak_efficiency_usd_per_mi,
        trips_assigned,
        ROUND(total_distance, 2) AS total_distance_mi,
        ROUND(total_revenue, 2) AS total_revenue_usd
    FROM local.warehouse.gold_assignment_efficiency 
    WHERE trips_assigned > 0  -- Filter out drivers who haven't completed a trip
    ORDER BY avg_fare_per_mile DESC 
    LIMIT 1
""").show()

--- Q: Which driver has the best fare-per-mile ratio? ---
+---------+-------------+--------------------------+--------------+-----------------+-----------------+
|driver_id|driver_rating|peak_efficiency_usd_per_mi|trips_assigned|total_distance_mi|total_revenue_usd|
+---------+-------------+--------------------------+--------------+-----------------+-----------------+
|     1104|        Spain|                      6.84|           229|           666.44|          4561.17|
+---------+-------------+--------------------------+--------------+-----------------+-----------------+



In [4]:
print("--- Q: How balanced is the workload distribution across drivers? ---")
spark.sql("""
    SELECT 
        COUNT(driver_id) as total_active_drivers,
        ROUND(STDDEV(trips_assigned), 2) as trips_stddev,
        MAX(trips_assigned) as max_trips,
        MIN(trips_assigned) as min_trips,
        ROUND(MAX(trips_assigned) / NULLIF(MIN(trips_assigned), 0), 2) as workload_ratio,
        SUM(CAST(is_overloaded AS INT)) as overloaded_driver_count
    FROM local.warehouse.gold_workload_balance
    WHERE is_active = true
""").show()

--- Q: How balanced is the workload distribution across drivers? ---
+--------------------+------------+---------+---------+--------------+-----------------------+
|total_active_drivers|trips_stddev|max_trips|min_trips|workload_ratio|overloaded_driver_count|
+--------------------+------------+---------+---------+--------------+-----------------------+
|                 679|        0.45|      230|      229|           1.0|                      0|
+--------------------+------------+---------+---------+--------------+-----------------------+



In [5]:
print("--- Actionable Insight: Overloaded Drivers ---")
spark.sql("""
    SELECT 
        driver_id, 
        driver_rating, 
        trips_assigned, 
        is_overloaded
    FROM local.warehouse.gold_workload_balance
    WHERE is_overloaded = true
    ORDER BY trips_assigned DESC
""").show()

--- Actionable Insight: Overloaded Drivers ---
+---------+-------------+--------------+-------------+
|driver_id|driver_rating|trips_assigned|is_overloaded|
+---------+-------------+--------------+-------------+
+---------+-------------+--------------+-------------+



In [6]:
print("--- Actionable Insight: Highest Utilized Drivers ---")
spark.sql("""
    WITH SystemAverage AS (
        SELECT AVG(trips_assigned) as avg_trips 
        FROM local.warehouse.gold_assignment_efficiency 
        WHERE is_active = true
    )
    SELECT 
        e.driver_id, 
        e.driver_rating, 
        e.trips_assigned,
        ROUND((e.trips_assigned / s.avg_trips) * 100 - 100, 1) AS percent_above_avg
    FROM local.warehouse.gold_assignment_efficiency e
    CROSS JOIN SystemAverage s
    WHERE e.is_active = true
      AND e.trips_assigned > (s.avg_trips * 1.10) -- 10% more than average
    ORDER BY e.trips_assigned DESC
""").show()

--- Actionable Insight: Highest Utilized Drivers ---
+---------+-------------+--------------+-----------------+
|driver_id|driver_rating|trips_assigned|percent_above_avg|
+---------+-------------+--------------+-----------------+
+---------+-------------+--------------+-----------------+

